In [1]:
import chipwhisperer as cw
import time, os
from Crypto.Cipher import AES
bitstream_path =  r'/home/sareeta/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit'
assert os.path.isfile(bitstream_path), f"Bitstream not found: {bitstream_path}"

# 2) Connect to the capture board (CWLite)
scope = cw.scope()
scope.default_setup()

# 3) Connect and Program the FPGA
print("Programming CW305 FPGA with:", bitstream_path)
target = cw.target(scope, cw.targets.CW305, bsfile=bitstream_path, force=True)

(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.54.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 8958019                   to 19435993                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 39999996                  to 29538459                 
scope.clock.adc_rate                     changed from 39999996.0                to 29538459.0               
scope.clock.freq_ct

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/sareeta/chipwhisperer/software/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


In [2]:
# Fixed test vector
KEY = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
PT  = bytes.fromhex("00112233445566778899aabbccddeeff")

# Software AES (ground truth)
sw_ct = AES.new(KEY, AES.MODE_ECB).encrypt(PT)

# Send to FPGA
target.fpga_write(target.REG_CRYPT_KEY, KEY)
target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)

hw_ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("=== NORMAL AES OPERATION ===")
print("Plaintext :", PT.hex())
print("Key       :", KEY.hex())
print("SW AES CT :", sw_ct.hex())
print("HW AES CT :", hw_ct.hex())
print("MATCH?    :", sw_ct == hw_ct)


=== NORMAL AES OPERATION ===
Plaintext : 00112233445566778899aabbccddeeff
Key       : 000102030405060708090a0b0c0d0e0f
SW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
HW AES CT : 69c4e0d86a7b0430d8cdb78070b4c55a
MATCH?    : True


In [3]:
print(scope.trigger.triggers)
scope.trigger.triggers = "tio4"

tio4


In [4]:
print(scope.glitch.clk_src)
print(scope.clock.clkgen_freq)
print(scope.clock.adc_freq)

target
7384615.384615385
29538459


In [3]:
def aes_encrypt_once():
    # fire AES
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    # small wait so ciphertext register updates even if trigger missed
    time.sleep(0.001)
    return bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

def glitch_and_read():
    # arm scope + glitch
    scope.arm()

    # launch AES (this generates the external trigger used by ext_single)
    target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    # capture waveform (optional but useful for debugging alignment)
    cap_timeout = scope.capture()
    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    if cap_timeout:
        return "no_trigger", ct, None

    tr = scope.get_last_trace()

    if ct == hw_ct:
        return "correct", ct, tr
    else:
        return "faulty", ct, tr


In [4]:
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 1
#scope.io.glitch_hp = False 
# scope.gain.gain = 46 
# scope.gain.mode = "high" 
# scope.adc.samples = 500 
# scope.adc.offset = 0 
# scope.adc.basic_mode = "rising_edge" 
# scope.clock.adc_src = "clkgen_x4" 
scope.clock.freq_ctr_src = "clkgen" 
scope.clock.adc_phase = 0 
scope.trigger.triggers = "tio4" 
scope.clock.extclk_freq = 10000000 
scope.clock.clkgen_mul = 5 
scope.clock.clkgen_div = 48 
scope.clock.clkgen_freq = 10000000 
scope.io.hs2 = "glitch" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.ext_offset = 0 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.width = 10 
scope.glitch.offset = -20 
#self.api.setParameter(['Glitch Module', 'Output Mode', 'Clock XORd']) 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")

Glitch ready.
Clock glitch engine armed (but harmless so far)


In [5]:
scope.io.hs2 = "glitch" 
#scope.trigger.triggers = "tio4" 
scope.glitch.clk_src = 'clkgen' 
scope.glitch.trigger_src ="ext_single" #"continuous" #change this depending on glitching desired "ext_single" "continuous" 
scope.glitch.output = "clock_xor" 
scope.glitch.repeat = 5 
print("Glitch ready.") 
print("Clock glitch engine armed (but harmless so far)")

Glitch ready.
Clock glitch engine armed (but harmless so far)


In [ ]:
Starting Glitch Loop...
FAULT! w=-48 off=-3 ct=b09583c26948a5e8ac98d1e6ef934b28 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-44 off=1 ct=69c4e0d8a57b0430dccdb78074b4c55a | Diff Bytes: 3 at Pos: [5, 9, 13]
FAULT! w=-42 off=-49 ct=da07ac3e0b3939bfa75fe2a377634fad | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-41 off=-49 ct=6944e0486a130420d84522802020c452 | Diff Bytes: 10 at Pos: [2, 4, 6, 8, 10, 11, 13, 14, 15, 16]
FAULT! w=-40 off=1 ct=69c4c0d86a7b2430d8cdf78070b4955a | Diff Bytes: 4 at Pos: [3, 7, 11, 15]
FAULT! w=-38 off=-49 ct=484080080a130020d804000020200010 | Diff Bytes: 15 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-35 off=-16 ct=007755f4df24c487324dcb949438e46b | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-30 off=-21 ct=69c4e0d86a3b0420d8cda68070b4c552 | Diff Bytes: 4 at Pos: [6, 8, 11, 16]
FAULT! w=-30 off=2 ct=69c4e04860320420c885260020a40412 | Diff Bytes: 12 at Pos: [4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-28 off=-49 ct=1a7630ab2e62329d7b094f4d0269309c | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-18 off=-49 ct=93ad097b40048cb320a4a5fc32e7f3e1 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-18 off=1 ct=b9cbbaad25bc5c228c66c8cb847260cc | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-15 off=14 ct=86c4c0586a7a72b8d8cc370070b4c55a | Diff Bytes: 9 at Pos: [1, 3, 4, 6, 7, 8, 10, 11, 12]
FAULT! w=-11 off=10 ct=72de3e7b44c93206490fcc3a4254e4ae | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-4 off=-48 ct=69c47dd86ae40430b3cdb78070b4c5e1 | Diff Bytes: 4 at Pos: [3, 6, 9, 16]
FAULT! w=-2 off=-47 ct=17dd9e66819e9cdc6c70ab5442cd46e9 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-2 off=-47 ct=61c4e0d8405a2418c0c4b58070b4c55a | Diff Bytes: 8 at Pos: [1, 5, 6, 7, 8, 9, 10, 11]
FAULT! w=-1 off=-48 ct=098ce0184a7b141078ed570050f4057a | Diff Bytes: 14 at Pos: [1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-1 off=-48 ct=6fd4c3a1a90db720da0191e311b6b035 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-1 off=-48 ct=0ff48321e9a5b708ba093183539e1075 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-1 off=33 ct=00775c70cf24cdd189ba5396df3ce663 | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-1 off=48 ct=0afc7cd83a74df4a332ced3c700266bf | Diff Bytes: 14 at Pos: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16]
FAULT! w=-1 off=48 ct=6ae81fd85f87df83e7069a431a35d384 | Diff Bytes: 15 at Pos: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
FAULT! w=-1 off=48 ct=57d69ad83a23df498171ede870025d93 | Diff Bytes: 14 at Pos: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16]
FAULT! w=-1 off=48 ct=823f4f22390699b1e73a68fc6696fbbf | Diff Bytes: 16 at Pos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

--- Scan Complete ---
Summary: {'correct': 23740, 'faulty': 25, 'no_trigger': 0}
Total records: 23765

In [7]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
widths = [x for x in range(-49, 1, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off
        
        for rep in range(N_PER_SETTING):
            # --- THE WORKING LOGIC FROM CODE 1 ---
            scope.arm()
            
            # Launch AES
            target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
            target.fpga_write(target.REG_CRYPT_GO, b"\x01")
            
            # Capture the hardware result
            cap_timeout = scope.capture()
            ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
            
            # Trace processing (from Code 2)
            tr = None
            if not cap_timeout:
                tr = np.array(scope.get_last_trace(), dtype=np.float32)
            # --------------------------------------

            # Labeling Logic & Fault Byte Analysis
            diff_bytes_count = 0
            diff_positions = []

            if cap_timeout:
                label = "no_trigger"
            elif ct == hw_ct:
                label = "correct"
            else:
                label = "faulty"
                
                # Analyze which specific bytes are different (1-indexed positions)
                # ZIP pairs up corresponding bytes from expected vs actual ciphertext
                diff_positions = [
                    i for i, (b_expected, b_actual) in enumerate(zip(hw_ct, ct)) 
                    if b_expected != b_actual
                ]
                diff_bytes_count = len(diff_positions)
                
                faults.append((w, off, ct.hex()))
                print(f"FAULT! w={w} off={off} ct={ct.hex()} | Diff Bytes: {diff_bytes_count} at Pos: {diff_positions}")

            # Store in Hits dictionary
            hits[label] += 1

            # Create the record (from Code 2)
            records.append({
                "width": w,
                "offset": off,
                "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
                "diff_bytes_count": diff_bytes_count,
                "diff_positions": diff_positions
            })

            # Store trace if it exists
            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=-48 off=-3 ct=84c47276a543ebf4f95cbf80229ac5fa | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
FAULT! w=-44 off=1 ct=752824a993a461835985ceccf1aaf8bd | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
FAULT! w=-41 off=1 ct=84c47276a543ebf4f95cbf80229ac5fa | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
FAULT! w=-39 off=-10 ct=ae3564dc919ca2636b9a950c3368c255 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
FAULT! w=-35 off=-16 ct=cad6dd78c8b7ad5d38eb0b350e70aa21 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
FAULT! w=-30 off=-21 ct=69c4e0d86a3b0430d8cdb78070b4c55a | Diff Bytes: 1 at Pos: [5]
FAULT! w=-30 off=2 ct=69c4e0d86a3b0420d8cda68060b4c552 | Diff Bytes: 5 at Pos: [5, 7, 10, 12, 15]
FAULT! w=-29 off=-49 ct=6a843a3bce5b7471676896f97dff8122 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15

In [8]:
import os, time
import numpy as np
import pandas as pd
from collections import Counter
from Crypto.Cipher import AES
from scipy import stats

# Fixed AES key already programmed in FPGA
KEY = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
aes_sw = AES.new(KEY, AES.MODE_ECB)

TARGET_PATTERN = (2, 5, 8, 15)   # your clean fault pattern, 0-indexed


def correct_ct_for_pt(pt):
    return aes_sw.encrypt(pt)


def run_glitch_once(pt, ext=0, w=-18, off=1.5, repeat=1):
    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = repeat

    scope.arm()

    target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    timeout = scope.capture()

    if timeout:
        return "no_trigger", None, None

    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    try:
        tr = np.array(scope.get_last_trace(), dtype=np.float32)
    except:
        tr = None

    return "ok", ct, tr


def diff_positions(correct, faulty):
    return tuple(i for i, (a, b) in enumerate(zip(correct, faulty)) if a != b)


def collect_random_plaintext_fault_data(
    n_trials=50000,
    ext=0,
    w=-18,
    off=1.5,
    repeat=1,
    save_prefix="random_pt_glitch"
):
    records = []
    clean_pairs = []
    traces = []
    trace_labels = []

    pattern_counts = Counter()
    hits = Counter()

    for i in range(n_trials):
        pt = os.urandom(16)
        correct = correct_ct_for_pt(pt)

        status, faulty, tr = run_glitch_once(
            pt, ext=ext, w=w, off=off, repeat=repeat
        )

        if status == "no_trigger":
            hits["no_trigger"] += 1
            records.append({
                "trial": i,
                "pt_hex": pt.hex(),
                "correct_ct_hex": correct.hex(),
                "faulty_ct_hex": "",
                "label": "no_trigger",
                "diff_positions": "",
                "diff_byte_count": 0,
                "ext_offset": ext,
                "width": w,
                "offset": off,
                "repeat": repeat,
            })
            continue

        if faulty == correct:
            label = "correct"
            pos = tuple()
            hits["correct"] += 1
        else:
            label = "faulty"
            pos = diff_positions(correct, faulty)
            pattern_counts[pos] += 1
            hits["faulty"] += 1

            if pos == TARGET_PATTERN:
                clean_pairs.append((pt, correct, faulty))

        records.append({
            "trial": i,
            "pt_hex": pt.hex(),
            "correct_ct_hex": correct.hex(),
            "faulty_ct_hex": faulty.hex(),
            "label": label,
            "diff_positions": str(pos),
            "diff_byte_count": len(pos),
            "ext_offset": ext,
            "width": w,
            "offset": off,
            "repeat": repeat,
        })

        if tr is not None:
            traces.append(tr)
            trace_labels.append(label)

        if (i + 1) % 1000 == 0:
            print(
                f"{i+1}/{n_trials} | "
                f"correct={hits['correct']} "
                f"faulty={hits['faulty']} "
                f"clean={len(clean_pairs)} "
                f"timeout={hits['no_trigger']}"
            )

    df = pd.DataFrame(records)
    df.to_csv(f"{save_prefix}_records.csv", index=False)

    np.savez_compressed(
        f"{save_prefix}_clean_pairs.npz",
        pt=np.array([list(x[0]) for x in clean_pairs], dtype=np.uint8),
        correct=np.array([list(x[1]) for x in clean_pairs], dtype=np.uint8),
        faulty=np.array([list(x[2]) for x in clean_pairs], dtype=np.uint8),
    )

    if len(traces) > 0:
        np.savez_compressed(
            f"{save_prefix}_traces.npz",
            traces=np.array(traces, dtype=np.float32),
            labels=np.array(trace_labels),
        )

    print("\nDONE")
    print("Hits:", hits)
    print("Clean target faults:", len(clean_pairs))
    print("\nTop fault patterns:")
    for p, c in pattern_counts.most_common(10):
        print(c, p)

    return df, clean_pairs, pattern_counts

In [11]:
df, clean_pairs, pattern_counts = collect_random_plaintext_fault_data(
    n_trials=50000,
    ext=0,
    w=-4,
    off=-48,
    repeat=1,
    save_prefix="randpt_ext0_w-18_off1p5"
)

1000/50000 | correct=911 faulty=89 clean=20 timeout=0
2000/50000 | correct=1848 faulty=152 clean=35 timeout=0
3000/50000 | correct=2779 faulty=221 clean=52 timeout=0
4000/50000 | correct=3703 faulty=297 clean=71 timeout=0
5000/50000 | correct=4635 faulty=365 clean=92 timeout=0
6000/50000 | correct=5570 faulty=430 clean=117 timeout=0
7000/50000 | correct=6523 faulty=477 clean=130 timeout=0
8000/50000 | correct=7466 faulty=534 clean=149 timeout=0
9000/50000 | correct=8406 faulty=594 clean=168 timeout=0
10000/50000 | correct=9345 faulty=655 clean=193 timeout=0
11000/50000 | correct=10266 faulty=734 clean=213 timeout=0
12000/50000 | correct=11208 faulty=792 clean=240 timeout=0
13000/50000 | correct=12143 faulty=857 clean=273 timeout=0
14000/50000 | correct=13059 faulty=941 clean=307 timeout=0
15000/50000 | correct=13986 faulty=1014 clean=326 timeout=0
16000/50000 | correct=14898 faulty=1102 clean=356 timeout=0
17000/50000 | correct=15847 faulty=1153 clean=377 timeout=0
18000/50000 | correc

In [12]:
def run_ttest_on_clean_pairs(clean_pairs):
    if len(clean_pairs) == 0:
        print("No clean faults found.")
        return

    correct = np.array([list(c) for _, c, _ in clean_pairs], dtype=np.uint8)
    faulty  = np.array([list(f) for _, _, f in clean_pairs], dtype=np.uint8)

    diff_matrix = np.bitwise_xor(correct, faulty).astype(np.float64)

    print("Clean samples:", diff_matrix.shape[0])
    print("Matrix shape :", diff_matrix.shape)

    print("\n1st-order t-test")
    max_t1 = 0

    for byte_pos in range(16):
        col = diff_matrix[:, byte_pos]

        if np.std(col) < 1e-10:
            t = 0.0
        else:
            t, _ = stats.ttest_1samp(col, 0.0)

        max_t1 = max(max_t1, abs(t))
        print(f"byte {byte_pos:2d}: t = {abs(t):.2f}")

    print("\nMax 1st-order t:", round(max_t1, 2))

    centered = diff_matrix - diff_matrix.mean(axis=0)
    second_order = centered ** 2

    print("\n2nd-order t-test")
    max_t2 = 0

    for byte_pos in range(16):
        col = second_order[:, byte_pos]

        if np.std(col) < 1e-10:
            t = 0.0
        else:
            t, _ = stats.ttest_1samp(col, 0.0)

        max_t2 = max(max_t2, abs(t))
        print(f"byte {byte_pos:2d}: t = {abs(t):.2f}")

    print("\nMax 2nd-order t:", round(max_t2, 2))


run_ttest_on_clean_pairs(clean_pairs)

Clean samples: 1124
Matrix shape : (1124, 16)

1st-order t-test
byte  0: t = 0.00
byte  1: t = 0.00
byte  2: t = 60.36
byte  3: t = 0.00
byte  4: t = 0.00
byte  5: t = 58.43
byte  6: t = 0.00
byte  7: t = 0.00
byte  8: t = 59.94
byte  9: t = 0.00
byte 10: t = 0.00
byte 11: t = 0.00
byte 12: t = 0.00
byte 13: t = 0.00
byte 14: t = 0.00
byte 15: t = 57.48

Max 1st-order t: 60.36

2nd-order t-test
byte  0: t = 0.00
byte  1: t = 0.00
byte  2: t = 36.65
byte  3: t = 0.00
byte  4: t = 0.00
byte  5: t = 37.85
byte  6: t = 0.00
byte  7: t = 0.00
byte  8: t = 38.86
byte  9: t = 0.00
byte 10: t = 0.00
byte 11: t = 0.00
byte 12: t = 0.00
byte 13: t = 0.00
byte 14: t = 0.00
byte 15: t = 37.63

Max 2nd-order t: 38.86


In [18]:
TARGET_PATTERN = (0, 7, 10, 13)   

In [19]:
df, clean_pairs, pattern_counts = collect_random_plaintext_fault_data(
    n_trials=50000,
    ext=0,
    w=-4,
    off=-48,
    repeat=1,
    save_prefix="randpt_ext0_w-18_off1p5"
)

1000/50000 | correct=885 faulty=115 clean=19 timeout=0
2000/50000 | correct=1781 faulty=219 clean=27 timeout=0
3000/50000 | correct=2681 faulty=319 clean=39 timeout=0
4000/50000 | correct=3603 faulty=397 clean=44 timeout=0
5000/50000 | correct=4514 faulty=486 clean=56 timeout=0
6000/50000 | correct=5435 faulty=565 clean=65 timeout=0
7000/50000 | correct=6352 faulty=648 clean=77 timeout=0
8000/50000 | correct=7266 faulty=734 clean=86 timeout=0
9000/50000 | correct=8178 faulty=822 clean=90 timeout=0
10000/50000 | correct=9087 faulty=913 clean=102 timeout=0
11000/50000 | correct=9992 faulty=1008 clean=111 timeout=0
12000/50000 | correct=10875 faulty=1125 clean=124 timeout=0
13000/50000 | correct=11748 faulty=1252 clean=133 timeout=0
14000/50000 | correct=12643 faulty=1357 clean=139 timeout=0
15000/50000 | correct=13525 faulty=1475 clean=152 timeout=0
16000/50000 | correct=14439 faulty=1561 clean=163 timeout=0
17000/50000 | correct=15350 faulty=1650 clean=171 timeout=0
18000/50000 | correc

In [20]:
run_ttest_on_clean_pairs(clean_pairs)

Clean samples: 445
Matrix shape : (445, 16)

1st-order t-test
byte  0: t = 36.00
byte  1: t = 0.00
byte  2: t = 0.00
byte  3: t = 0.00
byte  4: t = 0.00
byte  5: t = 0.00
byte  6: t = 0.00
byte  7: t = 35.06
byte  8: t = 0.00
byte  9: t = 0.00
byte 10: t = 37.61
byte 11: t = 0.00
byte 12: t = 0.00
byte 13: t = 34.89
byte 14: t = 0.00
byte 15: t = 0.00

Max 1st-order t: 37.61

2nd-order t-test
byte  0: t = 22.78
byte  1: t = 0.00
byte  2: t = 0.00
byte  3: t = 0.00
byte  4: t = 0.00
byte  5: t = 0.00
byte  6: t = 0.00
byte  7: t = 23.85
byte  8: t = 0.00
byte  9: t = 0.00
byte 10: t = 22.49
byte 11: t = 0.00
byte 12: t = 0.00
byte 13: t = 24.95
byte 14: t = 0.00
byte 15: t = 0.00

Max 2nd-order t: 24.95


In [21]:
TARGET_PATTERN = (0, 3, 11, 12) 
df, clean_pairs, pattern_counts = collect_random_plaintext_fault_data(
    n_trials=50000,
    ext=0,
    w=-25,
    off=-26,
    repeat=1,
    save_prefix="randpt_ext0_w-18_off1p5"
)

1000/50000 | correct=984 faulty=16 clean=0 timeout=0
2000/50000 | correct=1978 faulty=22 clean=0 timeout=0
3000/50000 | correct=2975 faulty=25 clean=0 timeout=0
4000/50000 | correct=3966 faulty=34 clean=0 timeout=0
5000/50000 | correct=4960 faulty=40 clean=0 timeout=0
6000/50000 | correct=5951 faulty=49 clean=0 timeout=0
7000/50000 | correct=6946 faulty=54 clean=0 timeout=0
8000/50000 | correct=7934 faulty=66 clean=0 timeout=0
9000/50000 | correct=8920 faulty=80 clean=0 timeout=0
10000/50000 | correct=9912 faulty=88 clean=0 timeout=0
11000/50000 | correct=10901 faulty=99 clean=0 timeout=0
12000/50000 | correct=11885 faulty=115 clean=0 timeout=0
13000/50000 | correct=12871 faulty=129 clean=0 timeout=0
14000/50000 | correct=13866 faulty=134 clean=0 timeout=0
15000/50000 | correct=14855 faulty=145 clean=0 timeout=0
16000/50000 | correct=15845 faulty=155 clean=0 timeout=0
17000/50000 | correct=16834 faulty=166 clean=0 timeout=0
18000/50000 | correct=17825 faulty=175 clean=0 timeout=0
19000

In [22]:
run_ttest_on_clean_pairs(clean_pairs)

No clean faults found.


In [25]:
TARGET_PATTERN = (4, 8, 12) 
df, clean_pairs, pattern_counts = collect_random_plaintext_fault_data(
    n_trials=50000,
    ext=0,
    w=-19,
    off=1,
    repeat=1,
    save_prefix="randpt_ext0_w-18_off1p5"
)
run_ttest_on_clean_pairs(clean_pairs)

1000/50000 | correct=990 faulty=10 clean=4 timeout=0
2000/50000 | correct=1983 faulty=17 clean=7 timeout=0
3000/50000 | correct=2973 faulty=27 clean=8 timeout=0
4000/50000 | correct=3964 faulty=36 clean=10 timeout=0
5000/50000 | correct=4951 faulty=49 clean=14 timeout=0
6000/50000 | correct=5939 faulty=61 clean=17 timeout=0
7000/50000 | correct=6927 faulty=73 clean=18 timeout=0
8000/50000 | correct=7917 faulty=83 clean=22 timeout=0
9000/50000 | correct=8902 faulty=98 clean=24 timeout=0
10000/50000 | correct=9882 faulty=118 clean=30 timeout=0
11000/50000 | correct=10869 faulty=131 clean=33 timeout=0
12000/50000 | correct=11856 faulty=144 clean=36 timeout=0
13000/50000 | correct=12842 faulty=158 clean=41 timeout=0
14000/50000 | correct=13820 faulty=180 clean=45 timeout=0
15000/50000 | correct=14805 faulty=195 clean=48 timeout=0
16000/50000 | correct=15792 faulty=208 clean=53 timeout=0
17000/50000 | correct=16779 faulty=221 clean=54 timeout=0
18000/50000 | correct=17768 faulty=232 clean=6

In [35]:
byte8_diffs  = [f[8]  ^ c[8]  for f, c in zip(clean_faulty, clean_correct)]
byte12_diffs = [f[12] ^ c[12] for f, c in zip(clean_faulty, clean_correct)]

print(f"Byte 8  differential — unique values: {set(byte8_diffs)}")
print(f"Byte 12 differential — unique values: {set(byte12_diffs)}")
print(f"Byte 4  differential — unique values: {len(set([f[4]^c[4] for f,c in zip(clean_faulty,clean_correct)]))} unique")

NameError: name 'clean_faulty' is not defined

In [ ]:
TARGET_PATTERN = (8) 
df, clean_pairs, pattern_counts = collect_random_plaintext_fault_data(
    n_trials=100000,
    ext=0,
    w=-1,
    off=47,
    repeat=5,
    save_prefix="randpt_ext0_w-18_off1p5"
)
run_ttest_on_clean_pairs(clean_pairs)

In [23]:
for _, c, f in clean_pairs[:10]:
    pos = [i for i,(a,b) in enumerate(zip(c,f)) if a!=b]
    print(pos)
from collections import Counter

patterns = Counter()

for _, c, f in clean_pairs:
    pos = tuple(i for i,(a,b) in enumerate(zip(c,f)) if a!=b)
    patterns[pos] += 1

print(patterns.most_common(20))
for _, c, f in clean_pairs[:5]:
    print(bytes(a ^ b for a,b in zip(c,f)).hex())
diffs = np.array([
    list(bytes(a ^ b for a, b in zip(c, f)))
    for _, c, f in clean_pairs
], dtype=np.uint8)

for b in [4, 8, 12]:
    vals, counts = np.unique(diffs[:, b], return_counts=True)
    print("byte", b)
    for v, c in zip(vals, counts):
        print(hex(v), c)
vals = []

for _, c, f in clean_pairs:
    vals.append((c[4], f[4], c[4] ^ f[4]))

for row in vals[:20]:
    print(
        f"correct={row[0]:02x}  "
        f"faulty={row[1]:02x}  "
        f"xor={row[2]:02x}"
    )
for _, c, f in clean_pairs[:10]:
    print(
        f"b8 : {c[8]:02x} -> {f[8]:02x}   xor={c[8]^f[8]:02x}",
        f" | b12 : {c[12]:02x} -> {f[12]:02x}   xor={c[12]^f[12]:02x}"
    )

[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[4, 8, 12]
[((4, 8, 12), 277)]
000000009d0000000400000004000000
00000000180000000400000004000000
00000000510000000400000004000000
00000000c10000000400000004000000
00000000be0000000400000004000000
byte 4
0xd 1
0x12 1
0x13 2
0x16 2
0x18 3
0x19 3
0x1a 1
0x1c 1
0x1d 5
0x22 2
0x23 3
0x24 2
0x25 1
0x28 3
0x2d 3
0x30 4
0x31 4
0x32 2
0x33 2
0x34 1
0x37 6
0x39 4
0x3a 3
0x3c 3
0x41 1
0x42 5
0x43 2
0x47 2
0x4a 1
0x4c 2
0x4d 6
0x4e 1
0x4f 3
0x51 4
0x52 2
0x53 2
0x55 3
0x57 1
0x59 3
0x5d 1
0x60 2
0x61 3
0x63 3
0x64 1
0x6b 1
0x6c 1
0x6e 2
0x72 5
0x74 1
0x77 2
0x79 3
0x7a 3
0x7c 2
0x7d 2
0x7e 2
0x7f 1
0x81 2
0x83 2
0x85 2
0x89 3
0x8a 2
0x91 2
0x92 7
0x93 4
0x95 4
0x96 2
0x97 3
0x99 2
0x9c 2
0x9d 1
0x9f 3
0xa2 1
0xa4 1
0xa6 2
0xa7 2
0xa8 2
0xab 3
0xad 2
0xaf 1
0xb2 5
0xb4 1
0xb7 2
0xbe 2
0xc1 4
0xc6 2
0xc7 3
0xc8 2
0xc9 3
0xca 4
0xcb 1
0xcf 2
0xd0 1
0xd2 1
0xd3 5
0xd4 1
0xda 3
0xdf 1
0xe0 5
0xe2 3
0xe3 2

In [36]:
candidates = [
    (-1,47)
]
from collections import Counter

def test_setting(w, off, ext_off=0, n=10000):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    # Ensure hw_ct is in bytes format for comparison
    hw_ct_bytes = bytes.fromhex(hw_ct.hex()) if isinstance(hw_ct, str) else bytes(hw_ct)
    hw_ct_hex = hw_ct_bytes.hex()

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct_hex])
    print("Faulty:", len(cts) - counts[hw_ct_hex])
    print("Top 10:")
    
    for ct, count in counts.most_common(10):
        if ct == hw_ct_hex:
            tag = "CORRECT"
            print(f"{count} {tag} {ct}")
        else:
            tag = "FAULT"
            # Convert hex string back to bytes for structural position checking
            ct_bytes = bytes.fromhex(ct)
            
            # Find 1-indexed byte differences
            diff_positions = [
                idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
                if b_expected != b_actual
            ]
            diff_bytes_count = len(diff_positions)
            
            print(f"{count} {tag} {ct} | Diff Bytes: {diff_bytes_count} at Pos: {diff_positions}")

for w, off in candidates:
    test_setting(w, off)


SETTING: w= -1 off= 47 ext= 0
Timeouts: 0
Correct: 9784
Faulty: 216
Top 10:
9784 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
179 FAULT 69c4e0d86a7b043088cdb78070b4c55a | Diff Bytes: 1 at Pos: [8]
12 FAULT 69c4e0d8a57b04308ccdb78074b4c55a | Diff Bytes: 3 at Pos: [4, 8, 12]
4 FAULT 847bff7606b6ebf4785cbfc6229a547f | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
2 FAULT 0a8a093b89328caa90e0a5bcf2abf3a1 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
2 FAULT 984ccf943f80ef319ae6528f4f58be40 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
2 FAULT 69c4e0d8a57b043080cdb78074b4c55a | Diff Bytes: 3 at Pos: [4, 8, 12]
1 FAULT 84c47276a543ebf4695cbf80229ac5fa | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
1 FAULT 1d5183d93964d8e4995d62f3c99a4799 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 84c46376a535ebf4b85cbf80229ac539 | Diff Bytes: 13 a

In [34]:
candidates = [
    (-18, 1.5, 0),    # your current working point
    (-18, 1.5, 1),    # one cycle later
    (-18, 1.5, 2),    # two cycles later
    (-18, 1.5, 3),
    (-18, 1.5, 4),
    (-18, 1.5, 5),
]
from collections import Counter

def test_setting(w, off, ext_off, n=10000):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    # Ensure hw_ct is in bytes format for comparison
    hw_ct_bytes = bytes.fromhex(hw_ct.hex()) if isinstance(hw_ct, str) else bytes(hw_ct)
    hw_ct_hex = hw_ct_bytes.hex()

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct_hex])
    print("Faulty:", len(cts) - counts[hw_ct_hex])
    print("Top 10:")
    
    for ct, count in counts.most_common(10):
        if ct == hw_ct_hex:
            tag = "CORRECT"
            print(f"{count} {tag} {ct}")
        else:
            tag = "FAULT"
            # Convert hex string back to bytes for structural position checking
            ct_bytes = bytes.fromhex(ct)
            
            # Find 1-indexed byte differences
            diff_positions = [
                idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
                if b_expected != b_actual
            ]
            diff_bytes_count = len(diff_positions)
            
            print(f"{count} {tag} {ct} | Diff Bytes: {diff_bytes_count} at Pos: {diff_positions}")

    
    pattern_counts = Counter()
    
    for ct, count in counts.items():
        if ct == hw_ct_hex:
            continue
    
        ct_bytes = bytes.fromhex(ct)
        diff_positions = tuple(
            idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
            if b_expected != b_actual
        )
        pattern_counts[diff_positions] += count
    
    print("\n Top fault patterns:")
    for pattern, count in pattern_counts.most_common(10):
        print(count, pattern)

for w, off, ext_off in candidates:
    test_setting(w, off,ext_off)


SETTING: w= -18 off= 1.5 ext= 0
Timeouts: 0
Correct: 9842
Faulty: 158
Top 10:
9842 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
36 FAULT 69c4e0d8a57b0430dccdb78074b4c55a | Diff Bytes: 3 at Pos: [4, 8, 12]
19 FAULT 1789ff7634b6eb01795c14a6233fe37f | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
17 FAULT 847bff7606b6ebf4785cbfc6229a547f | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
13 FAULT 84c47276a543ebf4f95cbf80229ac5fa | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
7 FAULT 0a8a093b89328caa90e0a5bcf2abf3a1 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
5 FAULT 938a093b40328caa90e0a5bcf2abf3a1 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
4 FAULT 938a19bb40329c1c90e0878ff2abe325 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
4 FAULT 84c46376a535ebf4e35cbf80229ac539 | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5

KeyboardInterrupt: 

In [138]:
import numpy as np
from scipy import stats

# ── Step 1: Extract clean (4,8,12) faulty ciphertexts ─────────────────────
hw_ct_bytes = bytes(hw_ct) if not isinstance(hw_ct, bytes) else hw_ct
hw_ct_hex   = hw_ct_bytes.hex()

clean_faulty  = []
clean_correct = []

for ct_hex, count in counts.items():
    if ct_hex == hw_ct_hex:
        continue                          # skip correct outputs

    ct_bytes = bytes.fromhex(ct_hex)

    diff_positions = tuple(
        i for i, (a, b) in enumerate(zip(hw_ct_bytes, ct_bytes))
        if a != b
    )

    if diff_positions == (4, 8, 12):      # only keep clean faults
        for _ in range(count):
            clean_faulty.append(list(ct_bytes))
            clean_correct.append(list(hw_ct_bytes))

print(f"Clean faulty ciphertexts: {len(clean_faulty)}")
print(f"Correct ciphertext:       {hw_ct_hex}")

# ── Step 2: Build differential matrix ─────────────────────────────────────
# Each row = one injection, each column = one byte position (0-15)
# Value = faulty_byte XOR correct_byte
diff_matrix = np.array([
    [a ^ b for a, b in zip(f, c)]
    for f, c in zip(clean_faulty, clean_correct)
], dtype=np.float64)

print(f"\nDifferential matrix shape: {diff_matrix.shape}")
print(f"(rows=injections, cols=byte positions)")

# ── Step 3: 1st order Welch t-test ────────────────────────────────────────
# Tests if mean of each byte's differential is different from zero
# Sufficient for bit faults and some byte faults
print("\n── 1st Order t-test ──────────────────────────────")
t_stats_1st = {}
for byte_pos in range(16):
    col = diff_matrix[:, byte_pos]
    if np.std(col) < 1e-10:
        t_stats_1st[byte_pos] = 0.0
        continue
    t_val, _ = stats.ttest_1samp(col, 0.0)
    t_stats_1st[byte_pos] = abs(t_val)

max_t1 = max(t_stats_1st.values())
best_byte_1st = max(t_stats_1st, key=t_stats_1st.get)

print(f"{'Byte':>6} {'t-statistic':>14} {'Exploitable':>12}")
print("-" * 36)
for bp, tv in t_stats_1st.items():
    marker = " ← PEAK" if bp == best_byte_1st else ""
    flag   = "YES" if tv > 4.5 else "no"
    print(f"{bp:>6} {tv:>14.2f} {flag:>12}{marker}")

print(f"\nMax 1st order t-statistic: {max_t1:.2f}")
print(f"Threshold:                  4.5")
print(f"Exploitable (1st order):    {'YES ✓' if max_t1 > 4.5 else 'NO — try 2nd order'}")

# ── Step 4: 2nd order Welch t-test ────────────────────────────────────────
# Needed for byte and multi-byte faults where 1st order fails
# Computes higher-order statistical moments
print("\n── 2nd Order t-test ──────────────────────────────")
diff_centered = diff_matrix - diff_matrix.mean(axis=0)
diff_sq       = diff_centered ** 2

t_stats_2nd = {}
for byte_pos in range(16):
    col = diff_sq[:, byte_pos]
    if np.std(col) < 1e-10:
        t_stats_2nd[byte_pos] = 0.0
        continue
    t_val, _ = stats.ttest_1samp(col, 0.0)
    t_stats_2nd[byte_pos] = abs(t_val)

max_t2 = max(t_stats_2nd.values())
best_byte_2nd = max(t_stats_2nd, key=t_stats_2nd.get)

print(f"{'Byte':>6} {'t-statistic':>14} {'Exploitable':>12}")
print("-" * 36)
for bp, tv in t_stats_2nd.items():
    marker = " ← PEAK" if bp == best_byte_2nd else ""
    flag   = "YES" if tv > 4.5 else "no"
    print(f"{bp:>6} {tv:>14.2f} {flag:>12}{marker}")

print(f"\nMax 2nd order t-statistic: {max_t2:.2f}")
print(f"Threshold:                  4.5")
print(f"Exploitable (2nd order):    {'YES ✓' if max_t2 > 4.5 else 'NO'}")

# ── Step 5: Cross-byte 2nd order (catches diagonal/column patterns) ────────
print("\n── Cross-byte 2nd Order t-test ───────────────────")
cross_max = 0.0
best_pair  = (0, 0)
for i in range(16):
    for j in range(i+1, 16):
        cross = diff_centered[:, i] * diff_centered[:, j]
        if np.std(cross) < 1e-10:
            continue
        t_val, _ = stats.ttest_1samp(cross, 0.0)
        if abs(t_val) > cross_max:
            cross_max = abs(t_val)
            best_pair  = (i, j)

print(f"Max cross-byte t-statistic: {cross_max:.2f}")
print(f"Best byte pair:             {best_pair}")
print(f"Exploitable (cross):        {'YES ✓' if cross_max > 4.5 else 'NO'}")

# ── Step 6: Final verdict ──────────────────────────────────────────────────
print("\n══ FINAL VERDICT ═════════════════════════════════")
print(f"Fault pattern:              bytes {{4, 8, 12}}")
print(f"Number of clean faults:     {len(clean_faulty)}")
print(f"1st order max t:            {max_t1:.2f}")
print(f"2nd order max t:            {max_t2:.2f}")
print(f"Cross-byte max t:           {cross_max:.2f}")
overall = max(max_t1, max_t2, cross_max)
print(f"Overall max t:              {overall:.2f}")
print(f"Threshold:                  4.5")
if overall > 4.5:
    print(f"\n✓ FAULT MODEL CONFIRMED")
    print(f"  Bytes {{4, 8, 12}} produce an EXPLOITABLE fault")
    print(f"  at parameters w=-18, offset=1.5, ext_offset=0")
    if max_t1 > 4.5:
        print(f"  Detected by: 1st order t-test")
    elif max_t2 > 4.5:
        print(f"  Detected by: 2nd order t-test")
    else:
        print(f"  Detected by: cross-byte 2nd order t-test")
else:
    print(f"\n✗ Not yet confirmed — t < 4.5")
    print(f"  Try collecting more clean faulty ciphertexts")
    print(f"  or adjust parameters to increase fault rate")

Clean faulty ciphertexts: 666
Correct ciphertext:       69c4e0d86a7b0430d8cdb78070b4c55a

Differential matrix shape: (666, 16)
(rows=injections, cols=byte positions)

── 1st Order t-test ──────────────────────────────
  Byte    t-statistic  Exploitable
------------------------------------
     0           0.00           no ← PEAK
     1           0.00           no
     2           0.00           no
     3           0.00           no
     4           0.00           no
     5           0.00           no
     6           0.00           no
     7           0.00           no
     8           0.00           no
     9           0.00           no
    10           0.00           no
    11           0.00           no
    12           0.00           no
    13           0.00           no
    14           0.00           no
    15           0.00           no

Max 1st order t-statistic: 0.00
Threshold:                  4.5
Exploitable (1st order):    NO — try 2nd order

── 2nd Order t-test ──────────

In [106]:
candidates = [
    (-18,1.5)
]
from collections import Counter

def test_setting(w, off, ext_off=0, n=100000):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    # Ensure hw_ct is in bytes format for comparison
    hw_ct_bytes = bytes.fromhex(hw_ct.hex()) if isinstance(hw_ct, str) else bytes(hw_ct)
    hw_ct_hex = hw_ct_bytes.hex()

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct_hex])
    print("Faulty:", len(cts) - counts[hw_ct_hex])
    print("Top 10:")
    
    for ct, count in counts.most_common(10):
        if ct == hw_ct_hex:
            tag = "CORRECT"
            print(f"{count} {tag} {ct}")
        else:
            tag = "FAULT"
            # Convert hex string back to bytes for structural position checking
            ct_bytes = bytes.fromhex(ct)
            
            # Find 1-indexed byte differences
            diff_positions = [
                idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
                if b_expected != b_actual
            ]
            diff_bytes_count = len(diff_positions)
            
            print(f"{count} {tag} {ct} | Diff Bytes: {diff_bytes_count} at Pos: {diff_positions}")

    
    pattern_counts = Counter()
    
    for ct, count in counts.items():
        if ct == hw_ct_hex:
            continue
    
        ct_bytes = bytes.fromhex(ct)
        diff_positions = tuple(
            idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
            if b_expected != b_actual
        )
        pattern_counts[diff_positions] += count
    
    print("\n Top fault patterns:")
    for pattern, count in pattern_counts.most_common(10):
        print(count, pattern)
    return counts, pattern_counts, hw_ct_hex, hw_ct_bytes

counts, pattern_counts, hw_ct_hex, hw_ct_bytes = test_setting(
    -18, 1.5, ext_off=0, n=100000
)

#for w, off in candidates:
    #test_setting(w, off)


SETTING: w= -18 off= 1.5 ext= 0
Timeouts: 0
Correct: 96920
Faulty: 3080
Top 10:
96920 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
666 FAULT 69c4e0d8a57b0430dccdb78074b4c55a | Diff Bytes: 3 at Pos: [4, 8, 12]
274 FAULT 84c47276a543ebf4f95cbf80229ac5fa | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
268 FAULT 1789ff7634b6eb01795c14a6233fe37f | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
246 FAULT 847bff7606b6ebf4785cbfc6229a547f | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
215 FAULT 0a8a093b89328caa90e0a5bcf2abf3a1 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
126 FAULT 938a093b40328caa90e0a5bcf2abf3a1 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
94 FAULT 84c46376a535ebf4e35cbf80229ac539 | Diff Bytes: 13 at Pos: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15]
90 FAULT 938a19bb40329c1c90e0878ff2abe325 | Diff Bytes: 16 at Pos: [0, 1, 2, 3,

In [122]:
import os, time
import numpy as np
from collections import Counter, defaultdict

# AES inverse S-box
INV_SBOX = [
0x52,0x09,0x6a,0xd5,0x30,0x36,0xa5,0x38,0xbf,0x40,0xa3,0x9e,0x81,0xf3,0xd7,0xfb,
0x7c,0xe3,0x39,0x82,0x9b,0x2f,0xff,0x87,0x34,0x8e,0x43,0x44,0xc4,0xde,0xe9,0xcb,
0x54,0x7b,0x94,0x32,0xa6,0xc2,0x23,0x3d,0xee,0x4c,0x95,0x0b,0x42,0xfa,0xc3,0x4e,
0x08,0x2e,0xa1,0x66,0x28,0xd9,0x24,0xb2,0x76,0x5b,0xa2,0x49,0x6d,0x8b,0xd1,0x25,
0x72,0xf8,0xf6,0x64,0x86,0x68,0x98,0x16,0xd4,0xa4,0x5c,0xcc,0x5d,0x65,0xb6,0x92,
0x6c,0x70,0x48,0x50,0xfd,0xed,0xb9,0xda,0x5e,0x15,0x46,0x57,0xa7,0x8d,0x9d,0x84,
0x90,0xd8,0xab,0x00,0x8c,0xbc,0xd3,0x0a,0xf7,0xe4,0x58,0x05,0xb8,0xb3,0x45,0x06,
0xd0,0x2c,0x1e,0x8f,0xca,0x3f,0x0f,0x02,0xc1,0xaf,0xbd,0x03,0x01,0x13,0x8a,0x6b,
0x3a,0x91,0x11,0x41,0x4f,0x67,0xdc,0xea,0x97,0xf2,0xcf,0xce,0xf0,0xb4,0xe6,0x73,
0x96,0xac,0x74,0x22,0xe7,0xad,0x35,0x85,0xe2,0xf9,0x37,0xe8,0x1c,0x75,0xdf,0x6e,
0x47,0xf1,0x1a,0x71,0x1d,0x29,0xc5,0x89,0x6f,0xb7,0x62,0x0e,0xaa,0x18,0xbe,0x1b,
0xfc,0x56,0x3e,0x4b,0xc6,0xd2,0x79,0x20,0x9a,0xdb,0xc0,0xfe,0x78,0xcd,0x5a,0xf4,
0x1f,0xdd,0xa8,0x33,0x88,0x07,0xc7,0x31,0xb1,0x12,0x10,0x59,0x27,0x80,0xec,0x5f,
0x60,0x51,0x7f,0xa9,0x19,0xb5,0x4a,0x0d,0x2d,0xe5,0x7a,0x9f,0x93,0xc9,0x9c,0xef,
0xa0,0xe0,0x3b,0x4d,0xae,0x2a,0xf5,0xb0,0xc8,0xeb,0xbb,0x3c,0x83,0x53,0x99,0x61,
0x17,0x2b,0x04,0x7e,0xba,0x77,0xd6,0x26,0xe1,0x69,0x14,0x63,0x55,0x21,0x0c,0x7d
]

TARGET_PATTERN = (4, 8, 12)

def aes_encrypt_read(pt):
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")
    time.sleep(0.001)
    return bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

def glitch_encrypt_read(pt, w=-18, off=1.5, ext_off=0):
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.ext_offset = ext_off
    scope.glitch.repeat = 1

    scope.arm()
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
    target.fpga_write(target.REG_CRYPT_GO, b"\x01")

    timeout = scope.capture()
    ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

    return timeout, ct

def diff_positions(c, cf):
    return tuple(i for i, (a, b) in enumerate(zip(c, cf)) if a != b)

def collect_clean_pairs(n_trials=100000, w=-18, off=1.5, ext_off=0):
    clean_pairs = []
    pattern_counts = Counter()
    faulty = 0
    timeouts = 0

    for i in range(n_trials):
        pt = os.urandom(16)

        correct_ct = aes_encrypt_read(pt)
        timeout, faulty_ct = glitch_encrypt_read(pt, w=w, off=off, ext_off=ext_off)

        if timeout:
            timeouts += 1
            continue

        if faulty_ct != correct_ct:
            faulty += 1
            pos = diff_positions(correct_ct, faulty_ct)
            pattern_counts[pos] += 1

            if pos == TARGET_PATTERN:
                clean_pairs.append((pt, correct_ct, faulty_ct))

        if (i + 1) % 10000 == 0:
            print(f"{i+1}/{n_trials} trials | clean={len(clean_pairs)} | faulty={faulty}")

    print("\nDone")
    print("Timeouts:", timeouts)
    print("Faulty:", faulty)
    print("Clean target faults:", len(clean_pairs))
    print("Top patterns:")
    for p, c in pattern_counts.most_common(10):
        print(c, p)

    return clean_pairs, pattern_counts

In [123]:
clean_pairs, pattern_counts = collect_clean_pairs(
    n_trials=100000,
    w=-18,
    off=1.5,
    ext_off=0
)

10000/100000 trials | clean=60 | faulty=297
20000/100000 trials | clean=115 | faulty=572
30000/100000 trials | clean=161 | faulty=844
40000/100000 trials | clean=219 | faulty=1126
50000/100000 trials | clean=268 | faulty=1416
60000/100000 trials | clean=330 | faulty=1694
70000/100000 trials | clean=397 | faulty=1996
80000/100000 trials | clean=453 | faulty=2285
90000/100000 trials | clean=526 | faulty=2579
100000/100000 trials | clean=589 | faulty=2857

Done
Timeouts: 0
Faulty: 2857
Clean target faults: 589
Top patterns:
1725 (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)
589 (4, 8, 12)
286 (0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15)
29 (0, 3, 4, 6, 7, 8, 9, 10, 12, 13)
24 (0, 1, 4, 5, 8, 9, 11, 12, 14)
22 (0, 2, 4, 5, 7, 8, 10, 12, 13, 15)
22 (2, 3, 4, 5, 6, 8, 9, 12, 15)
16 (0, 1, 2, 4, 5, 7, 8, 10, 11, 12, 13, 14, 15)
14 (3, 4, 6, 8, 9, 12)
10 (0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)


In [125]:
from collections import Counter

diff_counter = Counter()

for pt, c, cf in clean_pairs:
    diff = bytes(a ^ b for a, b in zip(c, cf))
    diff_counter[diff.hex()] += 1

print("Unique differentials:", len(diff_counter))

for d, cnt in diff_counter.most_common(20):
    print(cnt, d)

Unique differentials: 125
14 00000000950000000400000004000000
13 00000000c90000000400000004000000
10 000000004e0000000400000004000000
9 00000000be0000000400000004000000
9 00000000470000000400000004000000
9 00000000430000000400000004000000
9 00000000770000000400000004000000
8 00000000ca0000000400000004000000
8 00000000ff0000000400000004000000
8 00000000160000000400000004000000
8 000000007e0000000400000004000000
8 00000000890000000400000004000000
8 00000000ad0000000400000004000000
8 00000000180000000400000004000000
7 00000000190000000400000004000000
7 00000000d40000000400000004000000
7 00000000290000000400000004000000
7 00000000e20000000400000004000000
7 00000000360000000400000004000000
7 00000000230000000400000004000000


In [128]:
print(len(clean_pairs))

589


In [134]:
def entropy_from_counter(counter):
    total = sum(counter.values())
    probs = np.array([v / total for v in counter.values()], dtype=np.float64)
    return -np.sum(probs * np.log2(probs))

def rank_last_round_byte(clean_pairs, byte_pos):
    scores = []

    for k in range(256):
        deltas = []

        for pt, c, cf in clean_pairs:
            d = INV_SBOX[c[byte_pos] ^ k] ^ INV_SBOX[cf[byte_pos] ^ k]
            deltas.append(d)

        ctr = Counter(deltas)
        unique_count = len(ctr)
        most_common_count = ctr.most_common(1)[0][1]
        most_common_ratio = most_common_count / len(deltas)
        ent = entropy_from_counter(ctr)

        scores.append({
            "key_guess": k,
            "unique_deltas": unique_count,
            "most_common_ratio": most_common_ratio,
            "entropy": ent,
            "top_delta": ctr.most_common(1)[0][0],
        })

    # lower entropy / fewer unique deltas / higher repeated delta is better
    scores.sort(key=lambda x: (x["entropy"], x["unique_deltas"], -x["most_common_ratio"]))
    return scores

for byte_pos in TARGET_PATTERN:
    print(f"\n=== Byte {byte_pos} last-round key ranking ===")
    scores = rank_last_round_byte(clean_pairs, byte_pos)

    for rank, s in enumerate(scores[:10], start=1):
        print(
            f"Rank {rank:2d}: k=0x{s['key_guess']:02x}, "
            f"entropy={s['entropy']:.3f}, "
            f"unique={s['unique_deltas']}, "
            f"top_ratio={s['most_common_ratio']:.3f}, "
            f"top_delta=0x{s['top_delta']:02x}"
        )
    all_entropy = [s["entropy"] for s in scores]
    print("\nBest entropy:", min(all_entropy))
    print("Median entropy:", np.median(all_entropy))
    print("Worst entropy:", max(all_entropy))


=== Byte 4 last-round key ranking ===
Rank  1: k=0xc8, entropy=6.784, unique=125, top_ratio=0.020, top_delta=0x8a
Rank  2: k=0x1a, entropy=6.786, unique=126, top_ratio=0.020, top_delta=0x94
Rank  3: k=0xef, entropy=6.786, unique=127, top_ratio=0.022, top_delta=0x6b
Rank  4: k=0xa2, entropy=6.786, unique=127, top_ratio=0.022, top_delta=0x70
Rank  5: k=0x0b, entropy=6.788, unique=126, top_ratio=0.024, top_delta=0xde
Rank  6: k=0x0d, entropy=6.789, unique=126, top_ratio=0.020, top_delta=0xf8
Rank  7: k=0xb9, entropy=6.789, unique=125, top_ratio=0.019, top_delta=0x0e
Rank  8: k=0x5b, entropy=6.792, unique=125, top_ratio=0.020, top_delta=0xb3
Rank  9: k=0x97, entropy=6.793, unique=127, top_ratio=0.019, top_delta=0xf5
Rank 10: k=0x3c, entropy=6.793, unique=126, top_ratio=0.019, top_delta=0x50

Best entropy: 6.7840100445105085
Median entropy: 6.834779628504132
Worst entropy: 6.888987529972608

=== Byte 8 last-round key ranking ===
Rank  1: k=0x13, entropy=6.796, unique=125, top_ratio=0.029, 

In [110]:
from collections import Counter

diff_counter = Counter()

for f in clean_faulty:
    diff = bytes(a ^ b for a, b in zip(f, hw_ct_bytes))
    diff_counter[diff.hex()] += 1

print("Unique differentials:", len(diff_counter))
for d, c in diff_counter.most_common(10):
    print(c, d)

Unique differentials: 1
666 00000000cf0000000400000004000000


In [109]:
# Extract only the clean (4,8,12) faults
clean_faulty  = []
clean_correct = []

hw_ct_bytes = bytes.fromhex(hw_ct.hex()) if isinstance(hw_ct, str) else bytes(hw_ct)

for ct_hex, count in counts.items():
    if ct_hex == hw_ct_hex:
        continue
    ct_bytes = bytes.fromhex(ct_hex)
    diff_positions = tuple(
        idx for idx, (b_exp, b_act) 
        in enumerate(zip(hw_ct_bytes, ct_bytes))
        if b_exp != b_act
    )
    if diff_positions == (4, 8, 12):
        # Add this ciphertext 'count' times
        for _ in range(count):
            clean_faulty.append(ct_bytes)
            clean_correct.append(hw_ct_bytes)

print(f"Clean faulty ciphertexts collected: {len(clean_faulty)}")

# ── 1st order t-test ──────────────────────────────────────
import numpy as np
from scipy import stats

diff_matrix = np.array([
    [a ^ b for a, b in zip(f, c)] 
    for f, c in zip(clean_faulty, clean_correct)
], dtype=np.float64)

max_t1 = 0
for byte_pos in range(16):
    col = diff_matrix[:, byte_pos]
    if np.std(col) < 1e-10:
        continue
    t, _ = stats.ttest_1samp(col, 0.0)
    max_t1 = max(max_t1, abs(t))

print(f"1st order t-statistic: {max_t1:.2f}")
print(f"Threshold: 4.5")
print(f"Exploitable (1st order): {max_t1 > 4.5}")

# ── 2nd order t-test ──────────────────────────────────────
diff_centered = diff_matrix - diff_matrix.mean(axis=0)
diff_sq = diff_centered ** 2

max_t2 = 0
for byte_pos in range(16):
    col = diff_sq[:, byte_pos]
    if np.std(col) < 1e-10:
        continue
    t, _ = stats.ttest_1samp(col, 0.0)
    max_t2 = max(max_t2, abs(t))

print(f"2nd order t-statistic: {max_t2:.2f}")
print(f"Exploitable (2nd order): {max_t2 > 4.5}")

Clean faulty ciphertexts collected: 666
1st order t-statistic: 0.00
Threshold: 4.5
Exploitable (1st order): False
2nd order t-statistic: 0.00
Exploitable (2nd order): False


In [121]:
candidates = [
    (-18,1.5)
]
from collections import Counter

def test_setting(w, off, ext_off=1, n=10000):
    cts = []
    timeouts = 0

    scope.glitch.ext_offset = ext_off
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    # Ensure hw_ct is in bytes format for comparison
    hw_ct_bytes = bytes.fromhex(hw_ct.hex()) if isinstance(hw_ct, str) else bytes(hw_ct)
    hw_ct_hex = hw_ct_bytes.hex()

    for i in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    print("\nSETTING:", "w=", w, "off=", off, "ext=", ext_off)
    print("Timeouts:", timeouts)
    print("Correct:", counts[hw_ct_hex])
    print("Faulty:", len(cts) - counts[hw_ct_hex])
    print("Top 10:")
    
    for ct, count in counts.most_common(10):
        if ct == hw_ct_hex:
            tag = "CORRECT"
            print(f"{count} {tag} {ct}")
        else:
            tag = "FAULT"
            # Convert hex string back to bytes for structural position checking
            ct_bytes = bytes.fromhex(ct)
            
            # Find 1-indexed byte differences
            diff_positions = [
                idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
                if b_expected != b_actual
            ]
            diff_bytes_count = len(diff_positions)
            
            print(f"{count} {tag} {ct} | Diff Bytes: {diff_bytes_count} at Pos: {diff_positions}")

    
    pattern_counts = Counter()
    
    for ct, count in counts.items():
        if ct == hw_ct_hex:
            continue
    
        ct_bytes = bytes.fromhex(ct)
        diff_positions = tuple(
            idx for idx, (b_expected, b_actual) in enumerate(zip(hw_ct_bytes, ct_bytes))
            if b_expected != b_actual
        )
        pattern_counts[diff_positions] += count
    
    print("\n Top fault patterns:")
    for pattern, count in pattern_counts.most_common(10):
        print(count, pattern)

for w, off in candidates:
    test_setting(w, off)


SETTING: w= -18 off= 1.5 ext= 1
Timeouts: 0
Correct: 9723
Faulty: 277
Top 10:
9723 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
50 FAULT 6dc4e0d86a7b0430d8cdb78070b4c55a | Diff Bytes: 1 at Pos: [0]
26 FAULT 6dc419736a7b0430d8cdb71ba1b4935a | Diff Bytes: 6 at Pos: [0, 2, 3, 11, 12, 14]
24 FAULT 6dc405736a7b0430d8cd1b1ba1b4935a | Diff Bytes: 7 at Pos: [0, 2, 3, 10, 11, 12, 14]
17 FAULT 6cc418736a7be330d9aa1b1ba083615a | Diff Bytes: 11 at Pos: [0, 2, 3, 6, 8, 9, 10, 11, 12, 13, 14]
15 FAULT 007754f0df24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
12 FAULT 007754f0cf24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
9 FAULT 00775470cf24c5d18bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
8 FAULT 007754f4df24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
6 FAULT 6dc4e0736a7b0430d8cdb71ba

In [ ]:
SETTING: w= -18 off= 1.5 ext= 2
Timeouts: 0
Correct: 9760
Faulty: 240
Top 10:
9760 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
31 FAULT 69c4e0d86a330420d8c5a68060b4c552 | Diff Bytes: 6 at Pos: [5, 7, 9, 10, 12, 15]
22 FAULT 69c4e0d86a3b0430d8cdb78070b4c55a | Diff Bytes: 1 at Pos: [5]
22 FAULT 733c70a0d4633d1485501f44d9959961 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
19 FAULT b9cbbaad25bc5c228c66c8cb847260cc | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
18 FAULT 69c4e0d86a3b0420d8cda68070b4c552 | Diff Bytes: 4 at Pos: [5, 7, 10, 15]
10 FAULT 69c4e0d86a3b0430d8cdb68070b4c55a | Diff Bytes: 2 at Pos: [5, 10]
8 FAULT 69c4e0d86a3b0420d8cda68060b4c552 | Diff Bytes: 5 at Pos: [5, 7, 10, 12, 15]
8 FAULT 69c4e0586a330420c8c5268020b4c452 | Diff Bytes: 9 at Pos: [3, 5, 7, 8, 9, 10, 12, 14, 15]
7 FAULT a20f6d795df8bb5dd94834bd7242ccda | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

 Top fault patterns:
80 (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)
31 (5, 7, 9, 10, 12, 15)
22 (5,)
18 (5, 7, 10, 15)
17 (0, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15)
11 (3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15)
10 (5, 10)
8 (5, 7, 10, 12, 15)
8 (3, 5, 7, 8, 9, 10, 12, 14, 15)
5 (3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15)



SETTING: w= -18 off= 1.5 ext= 1
Timeouts: 0
Correct: 9723
Faulty: 277
Top 10:
9723 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a
50 FAULT 6dc4e0d86a7b0430d8cdb78070b4c55a | Diff Bytes: 1 at Pos: [0]
26 FAULT 6dc419736a7b0430d8cdb71ba1b4935a | Diff Bytes: 6 at Pos: [0, 2, 3, 11, 12, 14]
24 FAULT 6dc405736a7b0430d8cd1b1ba1b4935a | Diff Bytes: 7 at Pos: [0, 2, 3, 10, 11, 12, 14]
17 FAULT 6cc418736a7be330d9aa1b1ba083615a | Diff Bytes: 11 at Pos: [0, 2, 3, 6, 8, 9, 10, 11, 12, 13, 14]
15 FAULT 007754f0df24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
12 FAULT 007754f0cf24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
9 FAULT 00775470cf24c5d18bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
8 FAULT 007754f4df24c5918bba5396dd3ce663 | Diff Bytes: 16 at Pos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
6 FAULT 6dc4e0736a7b0430d8cdb71ba1b4c55a | Diff Bytes: 4 at Pos: [0, 3, 11, 12]

 Top fault patterns:
110 (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)
50 (0,)
29 (0, 2, 3, 11, 12, 14)
26 (0, 2, 3, 10, 11, 12, 14)
17 (0, 2, 3, 6, 8, 9, 10, 11, 12, 13, 14)
8 (0, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14)
6 (0, 3, 11, 12)
6 (1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14)
5 (0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14)
5 (1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14)

In [19]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
widths = [x for x in range(-49, 8, 1) if x != 0]
offsets = [x for x in range(-49, 49, 1) if x != 0]

print("Starting Glitch Loop...")

for w in widths:
    scope.glitch.width = w
    for off in offsets:
        scope.glitch.offset = off
        
        for rep in range(N_PER_SETTING):
            # --- THE WORKING LOGIC FROM CODE 1 ---
            scope.arm()
            
            # Launch AES
            target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
            target.fpga_write(target.REG_CRYPT_GO, b"\x01")
            
            # Capture the hardware result
            cap_timeout = scope.capture()
            ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
            
            # Trace processing (from Code 2)
            tr = None
            if not cap_timeout:
                tr = np.array(scope.get_last_trace(), dtype=np.float32)
            # --------------------------------------

            # Labeling Logic
            if cap_timeout:
                label = "no_trigger"
            elif ct == hw_ct:
                label = "correct"
            else:
                label = "faulty"
                faults.append((w, off, ct.hex()))
                print(f"FAULT! w={w} off={off} ct={ct.hex()}")

            # Store in Hits dictionary
            hits[label] += 1

            # Create the record (from Code 2)
            records.append({
                "width": w,
                "offset": off,
                "rep": rep,
                "label": label,
                "ct_hex": ct.hex(),
            })

            # Store trace if it exists
            if tr is not None:
                traces.append(tr)
                labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

Starting Glitch Loop...
FAULT! w=-49 off=-49 ct=6977e0d8df6b0430d8ddb7f770c8305a
FAULT! w=-48 off=-49 ct=934c093bd1328caa6be0a5ab72abc9a1
FAULT! w=-44 off=-5 ct=cad6dd78c8b7ad5d38eb0b350e70aa21
FAULT! w=-42 off=1 ct=847bff7606b6ebf4785cbfc6229a547f
FAULT! w=-37 off=-14 ct=692be082867b046a85cdb78d3ab4e000
FAULT! w=-28 off=-21 ct=63c18cb29d56d085c51700fd7ffaa9c2
FAULT! w=-4 off=-48 ct=11c457d86a08044652cd60807094c5db
FAULT! w=-3 off=-48 ct=74fc9244333e957b2549009fb3c0d743
FAULT! w=-2 off=-47 ct=dde667649d28b3cff8208a795aaf9675
FAULT! w=-2 off=-47 ct=90cce8bad6539fe3cb129ed41bc823ba
FAULT! w=-1 off=-48 ct=91f50636eaa7240fb10bf5955298197a
FAULT! w=-1 off=-48 ct=91f50636eaa7340fb10b28955298097a
FAULT! w=-1 off=48 ct=770b77224bfd994ae73aed416660e4bf
FAULT! w=-1 off=48 ct=6a0f1fd85fc59983e7a09a6c1a35d384
FAULT! w=-1 off=48 ct=57d686d83a23df49812ced3c700266bf
FAULT! w=-1 off=48 ct=f73f4f2239c599b1e7a07c8f66f67c91
FAULT! w=1 off=-20 ct=db4b6d6be60d16119b2e90c7cad074a4
FAULT! w=1 off=-3 ct=66451

In [ ]:
import numpy as np

# Configuration
N_PER_SETTING = 5
records = []
traces = []
labels = []
faults = []
hits = {"correct": 0, "faulty": 0, "no_trigger": 0}

# Range Setup
ext_offsets = range(0, 5)   # 1 to 100
widths = range(-49, 49, 1)
offsets = range(-49, 49, 1)

print("Starting Glitch Loop...")

for ext_off in ext_offsets:
    scope.glitch.ext_offset = ext_off

    for w in widths:
        scope.glitch.width = w

        for off in offsets:
            scope.glitch.offset = off

            for rep in range(N_PER_SETTING):

                scope.arm()

                target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
                target.fpga_write(target.REG_CRYPT_GO, b"\x01")

                cap_timeout = scope.capture()
                ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

                tr = None
                if not cap_timeout:
                    tr = np.array(scope.get_last_trace(), dtype=np.float32)

                if cap_timeout:
                    label = "no_trigger"

                elif ct == hw_ct:
                    label = "correct"

                else:
                    label = "faulty"
                    faults.append((ext_off, w, off, ct.hex()))
                    print(
                        f"FAULT! ext={ext_off} w={w} off={off} ct={ct.hex()}"
                    )

                hits[label] += 1

                records.append({
                    "ext_offset": ext_off,
                    "width": w,
                    "offset": off,
                    "rep": rep,
                    "label": label,
                    "ct_hex": ct.hex(),
                })

                if tr is not None:
                    traces.append(tr)
                    labels.append(label)

print("\n--- Scan Complete ---")
print("Summary:", hits)
print("Total records:", len(records))

In [ ]:
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

# Correct ciphertext reference
correct_ct = hw_ct  # bytes, length 16

def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

def changed_positions(diff_bytes):
    return tuple(i for i, v in enumerate(diff_bytes) if v != 0)

# -----------------------------
# Step 2: compute mask/diff
# -----------------------------
analysis_rows = []

for r in records:
    ct = bytes.fromhex(r["ct_hex"])

    diff = xor_bytes(correct_ct, ct)
    changed = changed_positions(diff)

    row = dict(r)
    row["diff_hex"] = diff.hex()
    row["changed_positions"] = changed
    row["num_changed_bytes"] = len(changed)
    row["pattern"] = ",".join(map(str, changed)) if changed else "none"

    analysis_rows.append(row)

df = pd.DataFrame(analysis_rows)

# Only faulty ciphertexts
fault_df = df[df["label"] == "faulty"].copy()

print("Total faulty:", len(fault_df))
print(fault_df[["ext", "width", "offset", "rep", "ct_hex", "diff_hex",
                "num_changed_bytes", "pattern"]].head(20))

In [ ]:
# -----------------------------
# Step 3: cluster by pattern
# -----------------------------
pattern_summary = (
    fault_df
    .groupby(["pattern", "num_changed_bytes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nMost repeated fault patterns:")
print(pattern_summary.head(30).to_string(index=False))

In [ ]:
# -----------------------------
# Step 4: repeated pattern per setting
# -----------------------------
setting_pattern_summary = (
    fault_df
    .groupby(["ext", "width", "offset", "pattern", "num_changed_bytes"])
    .size()
    .reset_index(name="count")
    .sort_values(["count", "num_changed_bytes"], ascending=[False, True])
)

print("\nBest repeated patterns per setting:")
print(setting_pattern_summary.head(50).to_string(index=False))

In [ ]:
# -----------------------------
# AES-DFA candidate patterns
# -----------------------------
dfa_candidates = setting_pattern_summary[
    setting_pattern_summary["num_changed_bytes"].isin([1, 2, 3, 4])
].copy()

print("\nPotential DFA-useful candidates:")
print(dfa_candidates.head(50).to_string(index=False))

In [ ]:
df.to_csv("all_glitch_records_with_diff.csv", index=False)
fault_df.to_csv("faults_only_with_diff.csv", index=False)
pattern_summary.to_csv("pattern_summary.csv", index=False)
setting_pattern_summary.to_csv("setting_pattern_summary.csv", index=False)

print("Saved CSV files.")

In [7]:
scope.arm()

target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")

timeout = scope.capture()

print("timeout:", timeout)
print("trig_count:", scope.adc.trig_count)
print("adc_freq:", scope.clock.adc_freq)
print("clkgen_freq:", scope.clock.clkgen_freq)

trigger_time = scope.adc.trig_count / scope.clock.adc_freq
aes_cycles = trigger_time * scope.clock.clkgen_freq

print("trigger_time:", trigger_time)
print("approx AES cycles:", aes_cycles)

timeout: False
trig_count: 4
adc_freq: 29538459
clkgen_freq: 7384615.384615385
trigger_time: 1.3541667830403747e-07
approx AES cycles: 1.0000000859375076


In [ ]:
from collections import Counter
import time
import pandas as pd

GOLDEN = hw_ct.hex()

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)
    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

def test_setting(w, off, ext=0, n=500):
    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    for rep in range(n):
        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        time.sleep(0.001)
        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

    counts = Counter(cts)

    faulty_counts = {ct: c for ct, c in counts.items() if ct != GOLDEN}

    if faulty_counts:
        top_fault_ct, top_fault_count = Counter(faulty_counts).most_common(1)[0]
        changed = diff_bytes(top_fault_ct)
    else:
        top_fault_ct = None
        top_fault_count = 0
        changed = []

    result = {
        "ext_offset": ext,
        "width": w,
        "offset": off,
        "trials": n,
        "valid_captures": len(cts),
        "timeouts": timeouts,
        "correct": counts[GOLDEN],
        "faulty": len(cts) - counts[GOLDEN],
        "top_fault_count": top_fault_count,
        "top_fault_ct": top_fault_ct,
        "changed_byte_count": len(changed),
        "changed_bytes": changed,
    }

    return result, counts


# Focus around your best point: w=-39, off=1, ext=0
results = []
all_counts = {}

ext_offsets = range(0, 10)
widths = range(-43, -34, 1)
offsets = range(-3, 6, 1)

N = 500

print("Starting focused repeatability scan...")

for ext in ext_offsets:
    for w in widths:
        for off in offsets:
            result, counts = test_setting(w, off, ext=ext, n=N)

            results.append(result)
            all_counts[(ext, w, off)] = counts

            if result["top_fault_count"] > 0:
                print(
                    f"ext={ext:2d} w={w:4d} off={off:3d} | "
                    f"correct={result['correct']:3d} faulty={result['faulty']:3d} "
                    f"top_fault={result['top_fault_count']:3d} "
                    f"changed_bytes={result['changed_byte_count']} "
                    f"ct={result['top_fault_ct']}"
                )

df = pd.DataFrame(results)

# Ranking: repeated fault first, then fewer changed bytes, then fewer timeouts
df_ranked = df.sort_values(
    by=["top_fault_count", "changed_byte_count", "timeouts"],
    ascending=[False, True, True]
)

print("\nTop candidate settings:")
print(df_ranked.head(20)[[
    "ext_offset",
    "width",
    "offset",
    "correct",
    "faulty",
    "timeouts",
    "top_fault_count",
    "changed_byte_count",
    "changed_bytes",
    "top_fault_ct"
]])

Starting focused repeatability scan...


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -43 off=  1 | correct=496 faulty=  4 top_fault=  2 changed_bytes=16 ct=3e2e6da849e42fe48eba628df89ab6f4


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -42 off=  1 | correct=498 faulty=  2 top_fault=  1 changed_bytes=16 ct=5daf05e2c1c819be6f7ddd69235d16d3


...

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -40 off=  1 | correct=497 faulty=  3 top_fault=  1 changed_bytes=16 ct=3e9636a8de5718f44e7ddca5bc5d428c


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


ext= 0 w= -39 off=  1 | correct=494 faulty=  6 top_fault=  1 changed_bytes=14 ct=aa230d22bdbb552ed80d102a70f74e5e


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:801) Partial reconfiguration for offset = 0 may not work


In [14]:
best = df_ranked.iloc[0]

ext = int(best["ext_offset"])
w = int(best["width"])
off = int(best["offset"])

print("Best setting:", ext, w, off)

result, counts = test_setting(w, off, ext=ext, n=3000)

print(result)

print("\nTop 20 ciphertexts:")
for ct, count in counts.most_common(20):
    tag = "CORRECT" if ct == GOLDEN else "FAULT"
    print(count, tag, ct, diff_bytes(ct) if ct != GOLDEN else [])

Best setting: 1 -38 1
{'ext_offset': 1, 'width': -38, 'offset': 1, 'trials': 3000, 'valid_captures': 3000, 'timeouts': 0, 'correct': 2988, 'faulty': 12, 'top_fault_count': 4, 'top_fault_ct': '6dc405736a7b0430d8cd1b1ba1b4935a', 'changed_byte_count': 7, 'changed_bytes': [0, 2, 3, 10, 11, 12, 14]}

Top 20 ciphertexts:
2988 CORRECT 69c4e0d86a7b0430d8cdb78070b4c55a []
4 FAULT 6dc405736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
2 FAULT 6dc4e0d86a7b0430d8cdb78070b4c55a [0]
1 FAULT 6dc418736a7b0430d8cd1b1ba1b4935a [0, 2, 3, 10, 11, 12, 14]
1 FAULT 2ef71aa7417fe3f7327dcb1b503864da [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007755f4df24c487324dcb949438e663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 6dc4e0736a7b0430d8cdb71ba1b4bf5a [0, 3, 11, 12, 14]
1 FAULT 007755fedf01c4873229cb845038e4db [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
1 FAULT 007754f0df24c5918bba5396dd3ce663 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [15]:
from collections import Counter
import time

GOLDEN = hw_ct.hex()

candidates = [
    (1, -38, 1),   # ext, width, offset
    (0, -40, 1),
    (2, -39, 1),
]

N = 5000

def diff_bytes(ct_hex, golden_hex=GOLDEN):
    ct = bytes.fromhex(ct_hex)
    golden = bytes.fromhex(golden_hex)

    return [i for i, (a, b) in enumerate(zip(ct, golden)) if a != b]

for ext, w, off in candidates:

    print("\n" + "="*80)
    print(f"TESTING ext={ext}, width={w}, offset={off}")
    print("="*80)

    scope.glitch.ext_offset = ext
    scope.glitch.width = w
    scope.glitch.offset = off
    scope.glitch.repeat = 1

    cts = []
    timeouts = 0

    start = time.time()

    for i in range(N):

        scope.arm()

        target.fpga_write(target.REG_CRYPT_TEXTIN, PT)
        target.fpga_write(target.REG_CRYPT_GO, b"\x01")

        timeout = scope.capture()

        if timeout:
            timeouts += 1
            continue

        ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)).hex()
        cts.append(ct)

        if (i + 1) % 500 == 0:
            print(f"{i+1}/{N}")

    elapsed = time.time() - start

    counts = Counter(cts)

    correct = counts[GOLDEN]
    faulty = len(cts) - correct

    print("\nSUMMARY")
    print("Elapsed:", round(elapsed, 2), "seconds")
    print("Timeouts:", timeouts)
    print("Correct:", correct)
    print("Faulty :", faulty)

    print("\nTOP 20 CIPHERTEXTS")

    for ct, count in counts.most_common(20):

        if ct == GOLDEN:
            print(f"{count:5d}  CORRECT  {ct}")
        else:
            changed = diff_bytes(ct)

            print(
                f"{count:5d}  FAULT    {ct} "
                f"changed_bytes={len(changed)} "
                f"indices={changed}"
            )


TESTING ext=1, width=-38, offset=1
500/5000
1000/5000
1500/5000
2000/5000
2500/5000
3000/5000
3500/5000
4000/5000
4500/5000
5000/5000

SUMMARY
Elapsed: 15.76 seconds
Timeouts: 0
Correct: 4952
Faulty : 48

TOP 20 CIPHERTEXTS
 4952  CORRECT  69c4e0d86a7b0430d8cdb78070b4c55a
   14  FAULT    6dc4e0d86a7b0430d8cdb78070b4c55a changed_bytes=1 indices=[0]
    6  FAULT    6dc419736a7b0430d8cdb71ba1b4935a changed_bytes=6 indices=[0, 2, 3, 11, 12, 14]
    4  FAULT    6dc405736a7b0430d8cd1b1ba1b4935a changed_bytes=7 indices=[0, 2, 3, 10, 11, 12, 14]
    3  FAULT    007754f0cf24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7b0430d9cd1b1ba0b4935a changed_bytes=8 indices=[0, 2, 3, 8, 10, 11, 12, 14]
    2  FAULT    007754f0df24c5918bba5396dd3ce663 changed_bytes=16 indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    2  FAULT    6cc418736a7be330d9aa1b1ba083615a changed_bytes=11 indices=[0, 2, 3, 6, 8, 9, 10